In [21]:

# agent imports
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# imports for rss feed tool
from get_rss import get_rss
import feedparser
import re
from datetime import datetime, timedelta
import time

from dotenv import load_dotenv

load_dotenv()

True

In [22]:
system_prompt = """You are an AI Newsagent, your job is to find AI specific news articles from rss feeds
"""

In [23]:
@tool
def get_rss_feeds(): #function used in main.py
    """This tool gets items from rss feeds"""
    keywords = ["ai","artificial intelligence", "machine learning", "robot", 
                "robots", "data science", "grok", "openai", "chatgpt", "claude", 
                "anthropic","LLM","Mistral","Gemini", "Deepseek","llama","codex"]
    # add other rss for other big news outlets itv, sky, maybe as well things like real python
    urls = [# Original
            "https://www.theguardian.com/uk/technology/rss",
            "https://www.theguardian.com/uk-news/rss",
            "https://rss.nytimes.com/services/xml/rss/nyt/Technology.xml",
            "https://www.marktechpost.com/feed/",
            "https://www.wired.com/feed/rss",
            "https://openai.com/blog/rss/",
            "https://news.microsoft.com/source/topics/ai/feed/",
            "https://feeds.bbci.co.uk/news/technology/rss.xml",

            # General AI News
            "https://openai.com/blog/rss.xml",
            "https://deepmind.google/blog/rss.xml",
            "https://www.anthropic.com/news/rss.xml",
            "https://www.technologyreview.com/topic/artificial-intelligence/feed/",
            "https://venturebeat.com/category/ai/feed/",
            "https://www.theverge.com/ai-artificial-intelligence/rss/index.xml",

            # Research-Focused
            "https://export.arxiv.org/rss/cs.AI",
            "https://export.arxiv.org/rss/cs.LG",
            "https://paperswithcode.com/rss",

            # Enterprise / Business AI
            "https://techcrunch.com/tag/artificial-intelligence/feed/",
            "https://aibusiness.com/rss.xml",
            "https://www.mckinsey.com/featured-insights/artificial-intelligence/rss",

            # Aggregators / Newsletters
            "https://importai.substack.com/feed",
            "https://www.bensbites.co/rss",
            "https://tldr.tech/ai/rss",
            ]
    
    # regex pattern with word boundaries for all keywords
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, keywords)) + r')\b', re.IGNORECASE)
    
    cutoff = datetime.now() - timedelta(days=3)
    
    all_results = []  # container for results
    
    for url in urls:
        feed = feedparser.parse(url)
        top_ten = feed.entries[:15]
        feed_results = []
    
    
        for e in top_ten:
            if "published_parsed" not in e:
                continue
    
            pub_date = datetime.fromtimestamp(time.mktime(e.published_parsed))
    
            if pub_date > cutoff and (pattern.search(e.title or "") or pattern.search(e.summary or "")):
                feed_results.append({
                    "title": e.title,
                    "link": e.link,
                    "published": e.published,
                    "summary": e.get("summary", ""),
                    "source": feed.feed.get("title", url)
                })
    
        if feed_results:
            #all_results[url] = feed_results
            all_results.extend(feed_results)

    return all_results

In [24]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_rss_feeds],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [25]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Can you find me some news articles")
config = {"configurable": {"thread_id": "1"}}# this is for memory
response = agent.invoke(
    {"messages": [question]},
    config
)

print(response['messages'][-1].content)

Here are AI-focused news articles from the RSS feed. Let me know if you’d like more from a specific region or topic.

- The big AI job swap: why white-collar workers are ditching their careers — The Guardian
  Link: https://www.theguardian.com/technology/2026/feb/11/big-ai-job-swap-white-collar-workers-ditching-their-careers
  Published: Wed, 11 Feb 2026 05:00:08 GMT
  Summary: Explores how AI job losses are driving white-collar workers to switch to traditional trades, and how some face pay cuts or changes to their vocation.

- Man behind satirical Crewkerne Gazette AI videos was jailed in UK for hate crimes — The Guardian
  Link: https://www.theguardian.com/uk-news/2026/feb/11/crewkerne-gazette-joshua-bonehill-paine-ai-videos-hate-crimes
  Published: Wed, 11 Feb 2026 17:17:35 GMT
  Summary: Channel 4 News identifies the owner of Crewkerne Gazette as Joshua Bonehill-Paine, known for AI-edited videos of UK politicians and past hate-crime convictions.

- Is it possible to develop AI with

In [26]:
print(response)

{'messages': [HumanMessage(content='Can you find me some news articles', additional_kwargs={}, response_metadata={}, id='acc75c05-3718-4cbe-8001-dd5dbe1fa412'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 278, 'prompt_tokens': 153, 'total_tokens': 431, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D8Q1FNAmav4SBtbORWWGX2IF3A126', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c51ce-7d4c-7c51-8da5-eedceaeeefd4-0', tool_calls=[{'name': 'get_rss_feeds', 'args': {}, 'id': 'call_Ws2lvIVEVPcQ1nfKMKnRqpgh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 278, '

In [27]:
question = HumanMessage(content="Can you tell me which article shows the biggest risk of AI?")
config = {"configurable": {"thread_id": "1"}}# this is for memory
response = agent.invoke(
    {"messages": [question]},
    config
)

print(response['messages'][-1].content)

My pick for the article that best illustrates the biggest AI risk is:

- Is a secure AI assistant possible? — MIT Technology Review
  Link: https://www.technologyreview.com/2026/02/11/1132768/is-a-secure-ai-assistant-possible/
  Why it shows the biggest risk: It focuses on systemic safety concerns as AI agents gain the ability to act in the real world, use tools, and interact with other systems. The piece discusses how even small mistakes or security gaps in autonomous agents can lead to outsized and widespread harms, which underpins the most fundamental, long-term risk of AI.

Other notable risk-focused articles from the list (for context):
- AI chatbots pose 'dangerous' risk when giving medical advice — BBC News
  Link: https://www.bbc.com/news/articles/c3093gjy2ero?at_medium=RSS&at_campaign=rss
  Highlights risk to individuals’ health when AI provides unreliable medical guidance.

- A.I. Personalizes the Internet but Takes Away Control — The New York Times
  Link: https://www.nytime

In [28]:
question = HumanMessage(content="What is the URL for that one?")
config = {"configurable": {"thread_id": "1"}}# this is for memory
response = agent.invoke(
    {"messages": [question]},
    config
)

print(response['messages'][-1].content)

https://www.technologyreview.com/2026/02/11/1132768/is-a-secure-ai-assistant-possible/


In [29]:
question = HumanMessage(content="Can you give me a short summary for a social media post")
config = {"configurable": {"thread_id": "1"}}# this is for memory
response = agent.invoke(
    {"messages": [question]},
    config
)

print(response['messages'][-1].content)

Here are a few short social-media-friendly options. Pick the one that fits your tone, or tell me your platform and audience and I’ll tailor it.

- Twitter/X style: Can we build a truly secure AI assistant? MIT Technology Review weighs safety gaps and real-world risks as AI agents start using tools and acting autonomously. A must-read on AI safety. https://www.technologyreview.com/2026/02/11/1132768/is-a-secure-ai-assistant-possible/ #AI #AISafety

- General audience: The biggest AI risk: security and safety as AI agents operate in the real world. MIT Technology Review examines whether secure AI assistants are even possible and why safeguards matter. https://www.technologyreview.com/2026/02/11/1132768/is-a-secure-ai-assistant-possible/

- LinkedIn/Professional: MIT Technology Review investigates whether a secure AI assistant is possible, highlighting safety concerns as AI agents gain tool access and real-world impact. Essential read for AI policy and risk management. https://www.technol